# Modified-NEAT: Flappy Bird on Colab

Runs the **normal** (`main.py`) or **sequential / attention** (`sequential.py`) Flappy Bird example from
[KshRaven/Modified-NEAT](https://github.com/KshRaven/Modified-NEAT), on CPU or CUDA, with **one model shared by several genera**
(`Population(genera=N)`) and a live **runlytics dashboard** (cards + charts) inside the notebook or in its own tab.

Run the cells top to bottom. Only the *Settings* cell needs editing.

## 1. Settings

In [ ]:
#@title Settings { run: "auto", display-mode: "form" }
MODEL             = "sequential"  #@param ["sequential", "normal"]
GENERA            = 2             #@param {type:"integer"}
GENOMES_PER_GENUS = 100           #@param {type:"integer"}
GENERATIONS       = 200           #@param {type:"integer"}
DEVICE            = "auto"        #@param ["auto", "cuda", "cpu"]
NEAT_BACKEND      = "cpu"         #@param ["cpu", "cuda"]
LOAD              = "false"       #@param {type:"string"}

# sequential model only
EMBED_SIZE        = 32            #@param {type:"integer"}
LAYERS            = 1             #@param {type:"integer"}
HEADS             = 1             #@param {type:"integer"}
LIVES             = 15            #@param {type:"integer"}
DIFFERENTIAL      = True          #@param {type:"boolean"}
DIFF_MODE         = "softmax"     #@param ["softmax", "energy"]
BRANCH_MODE       = "tied"        #@param ["tied", "free"]
SHARE_KV          = True          #@param {type:"boolean"}
NOISE_REG         = 0.25          #@param {type:"number"}

# dashboard / storage
PORT              = 6006          #@param {type:"integer"}
TUNNEL            = "none"        #@param ["none", "cloudflared"]
PERSIST_DIR       = ""            #@param {type:"string"}
REPO_URL          = "https://github.com/KshRaven/Modified-NEAT"  #@param {type:"string"}
REPO_BRANCH       = ""            #@param {type:"string"}
PATCH_ZIP         = ""            #@param {type:"string"}

# any other constant of the chosen example, e.g. {"GAP_SIZE": (140, 140)}; strings of code go through Raw("...")
EXTRA_OVERRIDES   = {}

**Notes**
- `GENERA = 1` keeps the example's original behaviour (`MODEL_NUM` separate models). `GENERA > 1` builds **one** model and
  `GENERA` genera of `GENOMES_PER_GENUS` genomes each.
- `DEVICE` is the PyTorch device that runs the model. `NEAT_BACKEND` is the evolution/bookkeeping backend; `cuda` needs
  `cupy` + `numba-cuda` (installed below on request) and silently falls back to `cpu` if unusable.
- `PERSIST_DIR` (e.g. `/content/drive/MyDrive/neat`) keeps checkpoints and logs across sessions. Mount Drive first.
- `PATCH_ZIP`: path of an uploaded zip whose files are copied over the clone. The public GitHub copy can be behind your working tree
  (for example it logs with plain TensorBoard instead of the runlytics logger), so `modified-neat-sync.zip` brings the clone up to date.

## 2. Install and import the repository

In [ ]:
import os, sys, subprocess, shutil, zipfile, time, socket, re, types, importlib

def sh(cmd, quiet=False):
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if not quiet or p.returncode: print(p.stdout[-3000:])
    return p.returncode

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = os.path.abspath("Modified-NEAT")

if not os.path.isdir(REPO_DIR):
    sh(f"git clone --depth 1 {'-b ' + REPO_BRANCH if REPO_BRANCH else ''} {REPO_URL} {REPO_DIR}")
else:
    print("repo already present:", REPO_DIR)

if PATCH_ZIP:
    with zipfile.ZipFile(PATCH_ZIP) as z:
        z.extractall(REPO_DIR)
    print(f"applied {PATCH_ZIP} over the clone")

# torch is preinstalled on Colab; everything else the examples use
sh("pip install -q numba matplotlib tensorboard gymnasium 'pygame-ce>=2.5.8' colorama runlytics", quiet=True)
if NEAT_BACKEND == "cuda":
    sh("pip install -q cupy-cuda12x 'numba-cuda[cu12]'", quiet=True)

# must be set before ModifiedNEAT / pygame are imported
os.environ["MODIFIEDNEAT_DEVICE"] = NEAT_BACKEND
os.environ["SDL_VIDEODRIVER"] = "dummy"        # no display on Colab
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["MODIFIEDNEAT_SKIP_DASHBOARD_BUILD"] = "1"
for p in (REPO_DIR, os.path.join(REPO_DIR, "examples", "flappy-bird")):
    if p not in sys.path: sys.path.insert(0, p)

import torch
TORCH_DEVICE = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
if TORCH_DEVICE == "cuda" and not torch.cuda.is_available():
    print("CUDA requested but not available (Runtime > Change runtime type > GPU); using cpu")
    TORCH_DEVICE = "cpu"
print("torch device:", TORCH_DEVICE, "| NEAT backend:", NEAT_BACKEND)

import ModifiedNEAT as neat
from ModifiedNEAT.util import storage
if PERSIST_DIR:
    os.makedirs(PERSIST_DIR, exist_ok=True)
    storage.set_storage_location(PERSIST_DIR.rstrip("/") + "/")
LOGDIR = os.path.join(storage.STORAGE_DIR, "neat_rl_logs")
os.makedirs(LOGDIR, exist_ok=True)
print("checkpoints / logs under:", storage.STORAGE_DIR)

# feature check: tells you if the clone lacks the shared-model (genera) example support
_src = open(os.path.join(REPO_DIR, "examples", "flappy-bird", "sequential.py"), errors="ignore").read()
if "GENERA" not in _src or not os.path.exists(os.path.join(REPO_DIR, "ModifiedNEAT", "rl", "logger.py")):
    print("WARNING: this clone lacks the shared-model (GENERA) example support and/or the runlytics logger.
"
          "         Push your current tree to GitHub, or upload modified-neat-sync.zip and set PATCH_ZIP.")

## 3. Example loader

The example scripts keep their settings as top-level constants, and some are used while the script is being imported
(for example the population size feeds the evolution config). So the loader rewrites the chosen constants in the
source and imports the result as a module. Nothing in the repository is modified.

In [ ]:
class Raw(str):
    """Code inserted verbatim as the value of a constant."""

def override_constants(src: str, values: dict, strict: bool = True):
    src = src.replace("\r\n", "\n")
    applied, skipped = [], []
    for name, value in values.items():
        pattern = re.compile(rf"^{name}[ \t]*=[^\n]*$", re.M)
        if not pattern.search(src):
            skipped.append(name)
            continue
        code = str(value) if isinstance(value, Raw) else repr(value)
        src = pattern.sub(lambda m: f"{name} = {code}  # notebook override", src, count=1)
        applied.append(name)
    if strict and skipped:
        print("not present in this example (ignored):", ", ".join(skipped))
    return src, applied

def build_overrides(model: str) -> dict:
    ov = {
        "DEVICE": Raw(f"torch.device('{TORCH_DEVICE}')"),
        "GENOMES": GENOMES_PER_GENUS,
        "GENERA": GENERA,
        "EPOCHS": GENERATIONS,
        "EMBED_SIZE": EMBED_SIZE,
        "LAYERS": LAYERS,
    }
    if model == "sequential":
        ov.update(HEADS=HEADS, LIVES=LIVES, DIFFERENTIAL=DIFFERENTIAL, DIFF_MODE=DIFF_MODE,
                  BRANCH_MODE=BRANCH_MODE, SHARE_KV=SHARE_KV, NOISE_REG=NOISE_REG)
    ov.update(EXTRA_OVERRIDES)
    return ov

def load_example(model: str):
    fname = {"sequential": "sequential.py", "normal": "main.py"}[model]
    path = os.path.join(REPO_DIR, "examples", "flappy-bird", fname)
    src, applied = override_constants(open(path, errors="ignore").read(), build_overrides(model))
    print("overrides applied:", ", ".join(applied))
    name = f"flappy_{model}"
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    exec(compile(src, path, "exec"), mod.__dict__)    # imports only; run() is called explicitly below
    return mod

## 4. Dashboard

Starts the runlytics server on `PORT` and shows it. **Colab**: embedded below the cell, plus a link that opens it in its own
tab. **Elsewhere**: an iframe on `localhost`. `TUNNEL = "cloudflared"` also creates a public `trycloudflare.com` link
(works anywhere, anyone with the link can view it). Charts fill in as soon as the first generation is logged.

In [ ]:
_dash = None
_tunnel = None

def start_dashboard(logdir=None, port=None, tunnel=None, height=760):
    global _dash, _tunnel
    logdir, port, tunnel = logdir or LOGDIR, port or PORT, tunnel or TUNNEL
    os.makedirs(logdir, exist_ok=True)
    if _dash is None or _dash.poll() is not None:
        _dash = subprocess.Popen(
            [sys.executable, "-m", "runlytics", "--logdir", logdir, "--port", str(port), "--reload_interval", "5"],
            stdout=open("/tmp/runlytics.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(60):                                   # wait until the port answers
        with socket.socket() as s:
            if s.connect_ex(("127.0.0.1", port)) == 0: break
        time.sleep(0.5)
    else:
        print("dashboard did not start:\n", open("/tmp/runlytics.log").read()[-1500:]); return
    print(f"runlytics serving {logdir} on port {port}")

    if tunnel == "cloudflared" and (_tunnel is None or _tunnel.poll() is not None):
        sh("wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared", quiet=True)
        _tunnel = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"],
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in _tunnel.stdout:
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m: print("public link:", m.group(0)); break

    if IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_iframe(port, height=height)    # inline
        output.serve_kernel_port_as_window(port)                   # link that opens its own tab
    else:
        from IPython.display import IFrame, display
        print(f"open http://localhost:{port} (or use the iframe)")
        display(IFrame(f"http://localhost:{port}", width="100%", height=height))

def stop_dashboard():
    for p in (_dash, _tunnel):
        if p is not None and p.poll() is None: p.terminate()

start_dashboard()

## 5. Train

Builds the environment, **one** model and the population (`GENERA` genera when `GENERA > 1`), then evolves for
`GENERATIONS`. The dashboard is shown at the top of this cell as well, so charts and the console output sit together.
Stop with the notebook's interrupt button; the run ends cleanly and keeps the checkpoint. Checkpoints are saved every
generation, so `LOAD = "true"` continues from the latest one.

In [ ]:
start_dashboard()                       # re-shows the dashboard above the training output (no-op if already running)
ex = load_example(MODEL)

argv = ["--train", "true"] + (["--load", LOAD] if LOAD.lower() not in ("false", "none", "") else [])
try:
    ex.run(argv=argv, test=False)       # test=False: the rendered test needs a window
except KeyboardInterrupt:
    print("\ninterrupted")

## 6. Results per genus

In [ ]:
pop = ex.POPULATION
print(f"generation {pop.generation} | genera {pop.genera} | models objects: {len({id(m) for m in pop.modules.values()})}")
for genus in pop.genera:
    top = sorted((g for g in pop.genomes.values() if g.genus == genus), key=lambda g: g.actual_fitness, reverse=True)[:5]
    print(f"\ngenus {genus}")
    for g in top:
        print(f"  genome {g.key:>6}   fitness {g.actual_fitness:.4f}")

## 7. Utilities (optional)
- Re-open the dashboard any time with `start_dashboard()`; `stop_dashboard()` shuts it down.
- Download checkpoints and logs as one zip:

In [ ]:
archive = shutil.make_archive("/content/neat_run" if IN_COLAB else "neat_run", "zip", storage.STORAGE_DIR)
print(archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)